# 05 · Fisher information selection (1D 2PL)

Does picking the most informative items beat random or anchor items on the 1D 2PL baseline?

- **Selectors:** random, IRT anchors, static Fisher, adaptive Fisher
- **Estimators:** accuracy, p-IRT, gp-IRT
- **Protocol:** same as the baseline (5 family splits, 20 seeds, k = 10…500). Items are fit on
  train, the gp-IRT blend on validation, and error is reported on test.

With $p_i=\sigma(\alpha_i\theta-\beta_i)$, item information is $I_i(\theta)=\alpha_i^2 p_i(1-p_i)$.
**Static** averages it over training abilities and takes the top k.
**Adaptive** asks one item, sees the answer, refits θ, and repeats.

Accuracy on hand-picked items is biased, so it is only an ablation here.
θ is fit with Newton MAP instead of the baseline's Adam (same answer, faster).

In [ ]:
import sys

sys.path.insert(0, "../scripts/irt")

import numpy as np
import pandas as pd
from efficbench import ROOT, load
from protocol import fit_context, make_splits
from run_progression import INFORMATION_SELECTORS, run_selection

from cabench.plotting import plot_score_errors

ks = [10, 20, 50, 100, 200, 500]
seeds = range(20)
out = ROOT / "results/irt/progression/information"

Y, models, items = load()
Y = Y.astype(np.float32)
splits = make_splits(Y)
subjects, model_ids, item_ids = items.subject.values, models.model_id.values, items.item_id.values

# Optional: one frozen fitted model per split, to skip fitting (see scripts/irt/README.md).
contexts = None

## Run

Fits one model per split, tunes gp-IRT on validation, then scores test.
Results go to `out`; `baseline.csv` is not touched.

In [ ]:
if contexts is None:
    contexts = [fit_context(Y[train], subjects, n_dims=1) for train, _, _ in splits]

metrics, records = run_selection(
    Y,
    splits,
    contexts,
    INFORMATION_SELECTORS,
    out,
    model_ids,
    ks=ks,
    seeds=seeds,
    item_ids=item_ids,
)

## Error by selector

Test MAE in percentage points, ±1 SD across splits. Compare selectors inside a panel;
compare panels to see what the estimator changes.

In [ ]:
plot_score_errors(records);

In [ ]:
100 * metrics.groupby(["k", "selector", "estimator"]).mae.mean().unstack("estimator")

## Paired change vs random + gp-IRT

Negative ΔMAE means lower error than the reference. The CI bootstraps the five split means (seeds averaged first). The splits overlap, so read it as sensitivity to the split, not as five independent samples.

In [ ]:
paired = pd.read_csv(out / "paired.csv")
paired = paired[paired.method.str.endswith("+ gp-IRT")].copy()
paired[["delta_mae", "ci_low", "ci_high"]] *= 100
paired[["method", "k", "delta_mae", "ci_low", "ci_high", "splits_won", "n_splits"]]

gp-IRT blend weight chosen on validation, per selector and k:

In [ ]:
pd.read_csv(out / "gp_weights.csv").groupby(["selector", "k"]).gp_weight.agg(["mean", "min", "max"])

Everything here is 1D. Notebook 06 tests more dimensions.